# Enhanced Predictive Maintenance of Distribution Transformers
## Using Gradient Boosting Ensemble with Explainable AI

**Conference:** IEOM Bangkok 2026

**Objective:** Beat baseline SVM accuracy of 97.25%

**Dataset:** 15,869 distribution transformers from Cauca Department, Colombia (2019-2020)

## 1. Setup and Imports

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve,
                             average_precision_score)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
import shap
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 2. Data Loading

In [5]:
# Load datasets
df_2019 = pd.read_excel('Dataset_Year_2019.xlsx')
df_2020 = pd.read_excel('Dataset_Year_2020.xlsx')

print(f"2019 Dataset: {df_2019.shape[0]:,} rows, {df_2019.shape[1]} columns")
print(f"2020 Dataset: {df_2020.shape[0]:,} rows, {df_2020.shape[1]} columns")

FileNotFoundError: [Errno 2] No such file or directory: 'Dataset_Year_2019.xlsx'

In [ ]:
# Display column names
print("\nColumn Names:")
for i, col in enumerate(df_2019.columns, 1):
    print(f"{i:2}. {col}")

In [ ]:
# Data info
print("\n=== 2019 DATA INFO ===")
df_2019.info()

In [ ]:
# Basic statistics
df_2019.describe().T

## 3. Data Exploration & Analysis

In [ ]:
# Identify target columns
target_2019 = [col for col in df_2019.columns if 'burned' in col.lower()][0]
target_2020 = [col for col in df_2020.columns if 'burned' in col.lower()][0]

print(f"Target 2019: {target_2019}")
print(f"Target 2020: {target_2020}")

# Target distribution
print(f"\n=== 2019 Target Distribution ===")
print(df_2019[target_2019].value_counts())
print(f"Failure Rate: {df_2019[target_2019].mean()*100:.2f}%")
print(f"Class Imbalance Ratio: {df_2019[target_2019].value_counts()[0]/df_2019[target_2019].value_counts()[1]:.1f}:1")

print(f"\n=== 2020 Target Distribution ===")
print(df_2020[target_2020].value_counts())
print(f"Failure Rate: {df_2020[target_2020].mean()*100:.2f}%")
print(f"Class Imbalance Ratio: {df_2020[target_2020].value_counts()[0]/df_2020[target_2020].value_counts()[1]:.1f}:1")

### 3.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 2019 Distribution
ax1 = axes[0]
colors = ['#2ecc71', '#e74c3c']
df_2019[target_2019].value_counts().plot(kind='pie', ax=ax1, colors=colors,
                                          autopct='%1.1f%%', startangle=90,
                                          explode=(0, 0.1))
ax1.set_title('2019 Transformer Failures\n(807 failures / 15,873 total)', fontweight='bold')
ax1.set_ylabel('')

# 2020 Distribution
ax2 = axes[1]
df_2020[target_2020].value_counts().plot(kind='pie', ax=ax2, colors=colors,
                                          autopct='%1.1f%%', startangle=90,
                                          explode=(0, 0.1))
ax2.set_title('2020 Transformer Failures\n(629 failures / 15,873 total)', fontweight='bold')
ax2.set_ylabel('')

# Year comparison
ax3 = axes[2]
years = ['2019', '2020']
failed = [df_2019[target_2019].sum(), df_2020[target_2020].sum()]
non_failed = [len(df_2019) - failed[0], len(df_2020) - failed[1]]
x = np.arange(len(years))
width = 0.35
bars1 = ax3.bar(x - width/2, non_failed, width, label='Operational', color='#2ecc71')
bars2 = ax3.bar(x + width/2, failed, width, label='Failed', color='#e74c3c')
ax3.set_ylabel('Number of Transformers')
ax3.set_title('Failure Comparison by Year', fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(years)
ax3.legend()
ax3.bar_label(bars1, padding=3)
ax3.bar_label(bars2, padding=3)

plt.tight_layout()
plt.savefig('fig1_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.2 Feature Distributions

In [ ]:
# Identify key columns
burn_col = [col for col in df_2019.columns if 'burn' in col.lower()][0]
power_col = 'POWER'
users_col = 'Number of users'
eens_col = [col for col in df_2019.columns if 'eens' in col.lower()][0]
location_col = 'LOCATION'

print(f"Burn Rate Column: {burn_col}")
print(f"EENS Column: {eens_col}")

In [ ]:
# Feature distributions for 2019
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Burn Rate Distribution
ax1 = axes[0, 0]
df_2019[df_2019[target_2019]==0][burn_col].hist(bins=50, ax=ax1, alpha=0.7, label='Operational', color='#2ecc71')
df_2019[df_2019[target_2019]==1][burn_col].hist(bins=50, ax=ax1, alpha=0.7, label='Failed', color='#e74c3c')
ax1.set_xlabel('Burn Rate [Failures/year]')
ax1.set_ylabel('Frequency')
ax1.set_title('Burn Rate Distribution by Failure Status', fontweight='bold')
ax1.legend()

# 2. Power Distribution
ax2 = axes[0, 1]
df_2019[df_2019[target_2019]==0][power_col].hist(bins=30, ax=ax2, alpha=0.7, label='Operational', color='#2ecc71')
df_2019[df_2019[target_2019]==1][power_col].hist(bins=30, ax=ax2, alpha=0.7, label='Failed', color='#e74c3c')
ax2.set_xlabel('Power [kVA]')
ax2.set_ylabel('Frequency')
ax2.set_title('Power Distribution by Failure Status', fontweight='bold')
ax2.legend()

# 3. Number of Users Distribution
ax3 = axes[0, 2]
df_2019[df_2019[target_2019]==0][users_col].hist(bins=30, ax=ax3, alpha=0.7, label='Operational', color='#2ecc71')
df_2019[df_2019[target_2019]==1][users_col].hist(bins=30, ax=ax3, alpha=0.7, label='Failed', color='#e74c3c')
ax3.set_xlabel('Number of Users')
ax3.set_ylabel('Frequency')
ax3.set_title('Users Distribution by Failure Status', fontweight='bold')
ax3.legend()

# 4. EENS Distribution (log scale)
ax4 = axes[1, 0]
df_2019[eens_col].apply(lambda x: np.log1p(x)).hist(bins=50, ax=ax4, color='#3498db')
ax4.set_xlabel('log(EENS + 1) [kWh]')
ax4.set_ylabel('Frequency')
ax4.set_title('Electric Power Not Supplied (Log Scale)', fontweight='bold')

# 5. Location Distribution
ax5 = axes[1, 1]
location_failure = df_2019.groupby(location_col)[target_2019].agg(['sum', 'count'])
location_failure['failure_rate'] = location_failure['sum'] / location_failure['count'] * 100
bars = ax5.bar(['Rural (0)', 'Urban (1)'], location_failure['failure_rate'], color=['#3498db', '#9b59b6'])
ax5.set_ylabel('Failure Rate (%)')
ax5.set_title('Failure Rate by Location', fontweight='bold')
ax5.bar_label(bars, fmt='%.2f%%', padding=3)

# 6. Boxplot of Burn Rate by Failure
ax6 = axes[1, 2]
df_2019.boxplot(column=burn_col, by=target_2019, ax=ax6)
ax6.set_xlabel('Failure Status (0=OK, 1=Failed)')
ax6.set_ylabel('Burn Rate')
ax6.set_title('Burn Rate by Failure Status', fontweight='bold')
plt.suptitle('')

plt.tight_layout()
plt.savefig('fig2_feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.3 Correlation Analysis

In [ ]:
# Correlation heatmap
numeric_cols = df_2019.select_dtypes(include=[np.number]).columns
corr_matrix = df_2019[numeric_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap (2019)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Correlation with target
print("\n=== Correlation with Target Variable ===")
target_corr = corr_matrix[target_2019].drop(target_2019).sort_values(ascending=False)
print(target_corr)

### 3.4 Categorical Features Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Type of clients
ax1 = axes[0]
client_failure = df_2019.groupby('Type of clients')[target_2019].agg(['sum', 'count'])
client_failure['failure_rate'] = client_failure['sum'] / client_failure['count'] * 100
client_failure.sort_values('failure_rate', ascending=False)['failure_rate'].plot(kind='bar', ax=ax1, color='#3498db')
ax1.set_xlabel('Type of Clients')
ax1.set_ylabel('Failure Rate (%)')
ax1.set_title('Failure Rate by Client Type', fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Type of installation
ax2 = axes[1]
install_failure = df_2019.groupby('Type of installation')[target_2019].agg(['sum', 'count'])
install_failure['failure_rate'] = install_failure['sum'] / install_failure['count'] * 100
install_failure.sort_values('failure_rate', ascending=False)['failure_rate'].plot(kind='bar', ax=ax2, color='#9b59b6')
ax2.set_xlabel('Type of Installation')
ax2.set_ylabel('Failure Rate (%)')
ax2.set_title('Failure Rate by Installation Type', fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('fig4_categorical_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
def preprocess_data(df, target_col):
    """Preprocess the dataset"""
    df_processed = df.copy()
    
    # Separate features and target
    X = df_processed.drop(columns=[target_col])
    y = df_processed[target_col]
    
    # Encode categorical variables
    label_encoders = {}
    for col in X.columns:
        if X[col].dtype == 'object':
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            label_encoders[col] = le
    
    return X, y, X.columns.tolist(), label_encoders

# Preprocess 2019 data
X_2019, y_2019, feature_names, label_encoders = preprocess_data(df_2019, target_2019)
print(f"Features shape: {X_2019.shape}")
print(f"Feature names: {feature_names}")

In [ ]:
# Feature Engineering
def engineer_features(X, feature_names):
    """Create engineered features"""
    X_eng = X.copy()
    
    # Find column indices
    burn_idx = [i for i, col in enumerate(feature_names) if 'burn' in col.lower()]
    eens_idx = [i for i, col in enumerate(feature_names) if 'eens' in col.lower()]
    users_idx = [i for i, col in enumerate(feature_names) if 'user' in col.lower()]
    power_idx = [i for i, col in enumerate(feature_names) if col.lower() == 'power']
    
    burn_col = feature_names[burn_idx[0]] if burn_idx else None
    eens_col = feature_names[eens_idx[0]] if eens_idx else None
    users_col = feature_names[users_idx[0]] if users_idx else None
    power_col = feature_names[power_idx[0]] if power_idx else None
    
    # Create interaction features
    if burn_col and eens_col:
        X_eng['burn_x_eens'] = X_eng[burn_col] * X_eng[eens_col]
    
    if burn_col and users_col:
        X_eng['burn_x_users'] = X_eng[burn_col] * X_eng[users_col]
    
    if power_col and users_col:
        X_eng['power_per_user'] = X_eng[power_col] / (X_eng[users_col] + 1)
    
    if burn_col:
        X_eng['burn_squared'] = X_eng[burn_col] ** 2
        X_eng['burn_log'] = np.log1p(X_eng[burn_col])
    
    return X_eng, X_eng.columns.tolist()

X_eng, eng_feature_names = engineer_features(X_2019, feature_names)
print(f"\nOriginal features: {len(feature_names)}")
print(f"After engineering: {len(eng_feature_names)}")
print(f"New features: {set(eng_feature_names) - set(feature_names)}")

## 5. Model Training & Comparison

In [ ]:
# Prepare data for training
# Combine 2019 and 2020 for more training data
df_2019_copy = df_2019.copy()
df_2020_copy = df_2020.copy()
df_2019_copy = df_2019_copy.rename(columns={target_2019: 'Burned'})
df_2020_copy = df_2020_copy.rename(columns={target_2020: 'Burned'})
df_combined = pd.concat([df_2019_copy, df_2020_copy], ignore_index=True)

print(f"Combined dataset: {df_combined.shape[0]:,} rows")

# Preprocess combined data
X, y, feature_names, _ = preprocess_data(df_combined, 'Burned')
X, feature_names = engineer_features(X, feature_names)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE for class imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"\nTraining set before SMOTE: {np.bincount(y_train)}")
print(f"Training set after SMOTE: {np.bincount(y_train_balanced)}")
print(f"Test set: {np.bincount(y_test)}")

In [ ]:
# Define models
models = {
    'SVM (Baseline)': SVC(kernel='rbf', probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
}

# Train and evaluate
results = []
trained_models = {}

print("Training models...\n")
print(f"{'Model':<25} | {'Accuracy':>10} | {'Precision':>10} | {'Recall':>10} | {'F1':>10} | {'ROC-AUC':>10}")
print("-" * 90)

for name, model in models.items():
    # Train
    model.fit(X_train_balanced, y_train_balanced)
    trained_models[name] = model
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Metrics
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    }
    results.append(metrics)
    
    print(f"{name:<25} | {metrics['Accuracy']:>10.4f} | {metrics['Precision']:>10.4f} | {metrics['Recall']:>10.4f} | {metrics['F1-Score']:>10.4f} | {metrics['ROC-AUC']:>10.4f}")

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print("\nDone!")

In [ ]:
# Results summary
print("\n" + "=" * 60)
print("MODEL COMPARISON RESULTS (Sorted by Accuracy)")
print("=" * 60)
print(results_df.to_string(index=False))

# Save results
results_df.to_csv('model_comparison_results.csv', index=False)
print("\nResults saved to: model_comparison_results.csv")

## 6. Stacking Ensemble Model

In [ ]:
# Create Stacking Ensemble
base_models = [
    ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)),
    ('lgbm', LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
]

stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)

print("Training Stacking Ensemble...")
stacking_clf.fit(X_train_balanced, y_train_balanced)

# Evaluate
y_pred_stack = stacking_clf.predict(X_test_scaled)
y_pred_proba_stack = stacking_clf.predict_proba(X_test_scaled)[:, 1]

stack_metrics = {
    'Model': 'Stacking Ensemble',
    'Accuracy': accuracy_score(y_test, y_pred_stack),
    'Precision': precision_score(y_test, y_pred_stack),
    'Recall': recall_score(y_test, y_pred_stack),
    'F1-Score': f1_score(y_test, y_pred_stack),
    'ROC-AUC': roc_auc_score(y_test, y_pred_proba_stack)
}

print(f"\nStacking Ensemble Results:")
for k, v in stack_metrics.items():
    if k != 'Model':
        print(f"  {k}: {v:.4f}")

## 7. Results Visualization

In [ ]:
# Add stacking to results
all_results = pd.concat([results_df, pd.DataFrame([stack_metrics])], ignore_index=True)
all_results = all_results.sort_values('Accuracy', ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Model Comparison Bar Chart
ax1 = axes[0, 0]
colors = ['#e74c3c' if 'SVM' in m else '#3498db' for m in all_results['Model']]
bars = ax1.barh(all_results['Model'], all_results['Accuracy'] * 100, color=colors)
ax1.axvline(x=97.25, color='red', linestyle='--', linewidth=2, label='Baseline (97.25%)')
ax1.set_xlabel('Accuracy (%)')
ax1.set_title('Model Performance Comparison', fontweight='bold', fontsize=12)
ax1.legend(loc='lower right')
for bar in bars:
    ax1.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{bar.get_width():.2f}%', va='center', fontsize=9)

# 2. Confusion Matrix (Best Model)
ax2 = axes[0, 1]
best_model_name = all_results.iloc[0]['Model']
if best_model_name == 'Stacking Ensemble':
    best_pred = y_pred_stack
else:
    best_pred = trained_models[best_model_name].predict(X_test_scaled)

cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['Operational', 'Failed'],
            yticklabels=['Operational', 'Failed'])
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')
ax2.set_title(f'Confusion Matrix ({best_model_name})', fontweight='bold', fontsize=12)

# 3. ROC Curves
ax3 = axes[1, 0]
for name, model in trained_models.items():
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        ax3.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

# Add stacking
fpr_stack, tpr_stack, _ = roc_curve(y_test, y_pred_proba_stack)
ax3.plot(fpr_stack, tpr_stack, linewidth=3, label=f'Stacking (AUC={stack_metrics["ROC-AUC"]:.3f})')

ax3.plot([0, 1], [0, 1], 'k--', label='Random')
ax3.set_xlabel('False Positive Rate')
ax3.set_ylabel('True Positive Rate')
ax3.set_title('ROC Curves Comparison', fontweight='bold', fontsize=12)
ax3.legend(loc='lower right', fontsize=8)

# 4. Metrics Comparison Radar
ax4 = axes[1, 1]
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
top_3 = all_results.head(3)
x = np.arange(len(metrics_to_plot))
width = 0.25

for i, (_, row) in enumerate(top_3.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    ax4.bar(x + i*width, values, width, label=row['Model'][:15])

ax4.set_ylabel('Score')
ax4.set_title('Top 3 Models - All Metrics', fontweight='bold', fontsize=12)
ax4.set_xticks(x + width)
ax4.set_xticklabels(metrics_to_plot, rotation=45)
ax4.legend(loc='lower right')
ax4.set_ylim([0.5, 1.0])

plt.tight_layout()
plt.savefig('fig5_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. SHAP Explainability Analysis

In [ ]:
# SHAP analysis for XGBoost
print("Computing SHAP values for XGBoost...")
xgb_model = trained_models['XGBoost']
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_scaled[:500])

# Create SHAP plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Summary plot
plt.sca(axes[0])
shap.summary_plot(shap_values, X_test_scaled[:500], feature_names=feature_names, show=False, max_display=15)
axes[0].set_title('SHAP Feature Importance', fontweight='bold')

# Bar plot
plt.sca(axes[1])
shap.summary_plot(shap_values, X_test_scaled[:500], feature_names=feature_names, plot_type='bar', show=False, max_display=15)
axes[1].set_title('Mean |SHAP| Value', fontweight='bold')

plt.tight_layout()
plt.savefig('fig6_shap_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Cross-Year Validation (Novel Contribution)

In [ ]:
# Train on 2019, Test on 2020 (Temporal Validation)
print("="*60)
print("CROSS-YEAR VALIDATION: Train on 2019, Test on 2020")
print("="*60)

# Preprocess
X_train_2019, y_train_2019, _, _ = preprocess_data(df_2019, target_2019)
X_train_2019, _ = engineer_features(X_train_2019, X_train_2019.columns.tolist())

X_test_2020, y_test_2020, _, _ = preprocess_data(df_2020, target_2020)
X_test_2020, _ = engineer_features(X_test_2020, X_test_2020.columns.tolist())

# Align columns
common_cols = list(set(X_train_2019.columns) & set(X_test_2020.columns))
X_train_2019 = X_train_2019[common_cols]
X_test_2020 = X_test_2020[common_cols]

# Scale
scaler_cv = StandardScaler()
X_train_cv = scaler_cv.fit_transform(X_train_2019)
X_test_cv = scaler_cv.transform(X_test_2020)

# SMOTE
smote_cv = SMOTE(random_state=42)
X_train_cv_balanced, y_train_cv_balanced = smote_cv.fit_resample(X_train_cv, y_train_2019)

# Train best models
cv_models = {
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

cv_results = []
print(f"\n{'Model':<20} | {'Accuracy':>10} | {'F1-Score':>10} | {'ROC-AUC':>10}")
print("-" * 60)

for name, model in cv_models.items():
    model.fit(X_train_cv_balanced, y_train_cv_balanced)
    y_pred = model.predict(X_test_cv)
    y_pred_proba = model.predict_proba(X_test_cv)[:, 1]
    
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test_2020, y_pred),
        'F1-Score': f1_score(y_test_2020, y_pred),
        'ROC-AUC': roc_auc_score(y_test_2020, y_pred_proba)
    }
    cv_results.append(metrics)
    print(f"{name:<20} | {metrics['Accuracy']:>10.4f} | {metrics['F1-Score']:>10.4f} | {metrics['ROC-AUC']:>10.4f}")

cv_results_df = pd.DataFrame(cv_results)
cv_results_df.to_csv('cross_year_validation_results.csv', index=False)

## 10. Final Summary

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY - IEOM BANGKOK 2026 CONFERENCE PAPER")
print("="*70)

baseline_accuracy = 97.25
best_accuracy = all_results['Accuracy'].max() * 100
improvement = best_accuracy - baseline_accuracy

print(f"\n{'Metric':<30} | {'Value':>15}")
print("-"*50)
print(f"{'Baseline SVM Accuracy':<30} | {baseline_accuracy:>14.2f}%")
print(f"{'Best Model Accuracy':<30} | {best_accuracy:>14.2f}%")
print(f"{'Improvement':<30} | {improvement:>+14.2f}%")
print(f"{'Best Model':<30} | {all_results.iloc[0]['Model']:>15}")

print(f"\n\n{'='*70}")
print("KEY CONTRIBUTIONS FOR PAPER:")
print("="*70)
print("1. Comprehensive comparison of 10 ML algorithms (vs only SVM in original)")
print("2. Feature engineering with interaction terms")
print("3. SMOTE for handling class imbalance (19:1 ratio)")
print("4. Stacking ensemble combining XGBoost, LightGBM, Random Forest")
print("5. SHAP-based explainability analysis")
print("6. Cross-year temporal validation (Train 2019 -> Test 2020)")
print(f"\nAccuracy improvement: {improvement:+.2f}% over baseline")

In [ ]:
# Save all results
all_results.to_csv('final_results.csv', index=False)
print("\nAll results saved successfully!")
print("\nGenerated files:")
print("  - model_comparison_results.csv")
print("  - cross_year_validation_results.csv")
print("  - final_results.csv")
print("  - fig1_target_distribution.png")
print("  - fig2_feature_distributions.png")
print("  - fig3_correlation_heatmap.png")
print("  - fig4_categorical_analysis.png")
print("  - fig5_model_comparison.png")
print("  - fig6_shap_analysis.png")